In [ ]:
import os
import sys
import yaml
sys.path.append(os.path.abspath('../src'))

from utils import *

# Load config (relative to notebooks/)
with open('../config.yml', 'r') as f:
    config = yaml.safe_load(f)

I0000 00:00:1776536573.278025   23426 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
# 1. Load saved ResNet50 training history
resnet_history_path = config['models']['resnet50']['history']
resnet_history = load_history(resnet_history_path)

# 2. Plot training curves
plot_learning_curves(resnet_history, title="Training Curves: ResNet50 (Phase 1 + Fine-Tuning)")

FileNotFoundError: [Errno 2] No such file or directory: '../../models/resnet50/resnet50_history.json'

In [ ]:
# 1. Load saved Residual Block model training history
residual_history_path = config['models']['residual_block']['history']
residual_history = load_history(residual_history_path)

# 2. Plot training curves
plot_learning_curves(residual_history, title="Training Curves: Residual Block CNN")

In [ ]:
# Load best model checkpoints for all approaches
model_scratch_path = config['models']['scratch']['checkpoint']
model_resnet_path = config['models']['resnet50']['checkpoint']
model_residual_path = config['models']['residual_block']['checkpoint']
model_scratch = keras.models.load_model(model_scratch_path)
model_resnet = keras.models.load_model(model_resnet_path)
model_residual = keras.models.load_model(model_residual_path)

In [ ]:
# All paths relative to notebooks/
train_dir = config['paths']['train_dir']
val_dir = config['paths']['val_dir']
test_dir = config['paths']['test_dir']
IMG_SIZE = tuple(config['img_size'])
BATCH_SIZE = config['batch_size']

_, _, test_ds, class_names = load_datasets(
    train_dir, val_dir, test_dir,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

# Evaluate all models on the same test set
metrics_cnn      = evaluate_model(model_scratch, test_ds, class_names, model_name="CNN Scratch")
metrics_resnet   = evaluate_model(model_resnet,  test_ds, class_names, model_name="ResNet50 Fine-tuned")
metrics_residual = evaluate_model(model_residual, test_ds, class_names, model_name="Residual Block CNN")

compare_models([metrics_cnn, metrics_resnet, metrics_residual])